In [0]:
try:
    dbutils.widgets.get("LOAD_TYPE")
except:
    dbutils.widgets.dropdown(
        "LOAD_TYPE",
        "INCREMENTAL",
        ["FULL", "INCREMENTAL"],
        "Load Type"
    )

LOAD_TYPE = dbutils.widgets.get("LOAD_TYPE").upper()

print("Selected LOAD_TYPE:", LOAD_TYPE)

from pyspark.sql.functions import * 
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# --------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------

SILVER_TABLE = "agriculture_dev.silver.agriculture_clean"
GOLD_SCHEMA = "agriculture_dev.gold"
FACT_TABLE = "agriculture_dev.gold.fact_agriculture"


# --------------------------------------------------
# 2. READ VALID SILVER DATA
# --------------------------------------------------

silver_df = (
    spark.table(SILVER_TABLE)
    .filter(
        col("validation_errors").isNull() |
        (col("validation_errors") == "")
    )
)

print("Valid Silver records:", silver_df.count())

# --------------------------------------------------
# 3. DIMENSION CONFIGURATION
# --------------------------------------------------

# Value dimensions carry no changing attributes, so SCD1 on them
# reduces to an insert-only MERGE (nothing to overwrite).
value_dimension_config = {
    "dim_crop": ("crop_type", "crop_key"),
    "dim_season": ("season", "season_key"),
    "dim_irrigation": ("irrigation_type", "irrigation_key"),
    "dim_soil": ("soil_type", "soil_key")
}

# --------------------------------------------------
# 4. dim_farm - SCD TYPE 1
#    Keeps the LATEST observed crop/season per farm.
#    When an attribute changes the row is updated in
#    place (no history kept), with updated_ts bumped.
# --------------------------------------------------

farm_window = Window.partitionBy("farm_id").orderBy(
    col("row_hash").desc(),
    col("batch_id").desc()
)

dim_farm_df = (
    silver_df
    .select("farm_id", "crop_type", "season", "ingestion_timestamp", "batch_id","row_hash")
    .withColumn("rn", row_number().over(farm_window))
    .filter(col("rn") == 1)
    .select(
        col("farm_id"),
        col("crop_type").alias("latest_crop_type"),
        col("season").alias("latest_season"),
        current_timestamp().alias("created_ts"),
        current_timestamp().alias("updated_ts")
    )
)

if LOAD_TYPE == "FULL":

    # Sequential surrogate key: 1..N ordered by farm_id
    dim_farm_full = dim_farm_df.select(
        row_number().over(Window.orderBy(col("farm_id"))).cast("int").alias("farm_key"),
        col("farm_id"),
        col("latest_crop_type"),
        col("latest_season"),
        col("created_ts"),
        col("updated_ts")
    )

    dim_farm_full.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{GOLD_SCHEMA}.dim_farm")

    print("dim_farm created (SCD1):", spark.table(f"{GOLD_SCHEMA}.dim_farm").count())

elif LOAD_TYPE == "INCREMENTAL":

    # The target table may have been created with an older schema that
    # lacks the SCD1 attribute columns; add them so the MERGE can
    # resolve them (existing rows are backfilled with NULL).
    existing_cols = set(spark.table(f"{GOLD_SCHEMA}.dim_farm").columns)
    missing_cols = {
        "latest_crop_type": "STRING",
        "latest_season": "STRING",
        "created_ts": "TIMESTAMP",
        "updated_ts": "TIMESTAMP"
    }
    cols_to_add = [c for c in missing_cols if c not in existing_cols]
    if cols_to_add:
        add_spec = ", ".join(f"{c} {missing_cols[c]}" for c in cols_to_add)
        spark.sql(
            f"ALTER TABLE {GOLD_SCHEMA}.dim_farm ADD COLUMNS ({add_spec})"
        )

    # Assign sequential keys ONLY to farms not yet in the target, so
    # numbering continues from the current max farm_key and existing
    # keys stay stable (SCD1 updates never rewrite keys).
    farm_key_offset = (
        spark.table(f"{GOLD_SCHEMA}.dim_farm")
        .agg(coalesce(max("farm_key"), lit(0)))
        .collect()[0][0]
    )
    dim_farm_new = (
        dim_farm_df
        .join(
            spark.table(f"{GOLD_SCHEMA}.dim_farm").select("farm_id"),
            on="farm_id",
            how="left_anti"
        )
        .select(
            col("farm_id"),
            (lit(farm_key_offset)
             + row_number().over(Window.orderBy(col("farm_id")))).cast("int").alias("farm_key")
        )
    )
    dim_farm_source = dim_farm_df.join(dim_farm_new, on="farm_id", how="left")

    # Load the DeltaTable handle AFTER any ALTER so the MERGE sees
    # the current schema
    farm_target = DeltaTable.forName(spark, f"{GOLD_SCHEMA}.dim_farm")

    (
        farm_target.alias("target")
        .merge(
            dim_farm_source.alias("source"),
            "target.farm_id = source.farm_id"
        )
        # SCD1: overwrite attributes in place only when they changed
        # (null-safe so pre-existing rows backfilled with NULL are updated).
        # farm_key is NOT updated: existing keys stay stable, and only
        # new farms (from dim_farm_new) carry a newly assigned key.
        .whenMatchedUpdate(
            set={
                "latest_crop_type": "source.latest_crop_type",
                "latest_season": "source.latest_season",
                "updated_ts": "current_timestamp()"
            },
            condition=(
                "NOT (target.latest_crop_type != source.latest_crop_type)"
                " OR NOT (target.latest_season != source.latest_season)"
            )
        )
        .whenNotMatchedInsert(
            values={
                "farm_key": "source.farm_key",
                "farm_id": "source.farm_id",
                "latest_crop_type": "source.latest_crop_type",
                "latest_season": "source.latest_season",
                "created_ts": "source.created_ts",
                "updated_ts": "source.updated_ts"
            }
        )
        .execute()
    )

    print("dim_farm SCD1 MERGE completed:", spark.table(f"{GOLD_SCHEMA}.dim_farm").count())

# --------------------------------------------------
# 5. VALUE DIMENSIONS FULL / INCREMENTAL LOAD
# --------------------------------------------------

if LOAD_TYPE == "FULL":

    print("Starting Gold FULL load")

    for table_name, (business_col, key_col) in value_dimension_config.items():

        target_table = f"{GOLD_SCHEMA}.{table_name}"

        dim_df = (
            silver_df
            .select(business_col)
            .filter(
                col(business_col).isNotNull() &
                (trim(col(business_col)) != "")
            )
            .distinct()
            .select(
                row_number().over(Window.orderBy(col(business_col))).cast("int").alias(key_col),
                col(business_col)
            )
        )

        dim_df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(
            table_name,
            "created:",
            spark.table(target_table).count()
        )

elif LOAD_TYPE == "INCREMENTAL":

    print("Starting Gold INCREMENTAL dimension load")

    for table_name, (business_col, key_col) in value_dimension_config.items():

        target_table = f"{GOLD_SCHEMA}.{table_name}"

        incoming_df = (
            silver_df
            .select(business_col)
            .filter(
                col(business_col).isNotNull() &
                (trim(col(business_col)) != "")
            )
            .distinct()
        )

        existing_df = spark.table(target_table)

        # Continue the sequential numbering from the current max key;
        # assign keys ONLY to values not yet in the target so existing
        # keys stay stable. Matched rows carry a NULL key but are never
        # inserted (insert-only MERGE).
        key_offset = (
            existing_df.agg(coalesce(max(key_col), lit(0))).collect()[0][0]
        )
        new_values = (
            incoming_df
            .join(existing_df.select(business_col), on=business_col, how="left_anti")
            .select(
                col(business_col),
                (lit(key_offset)
                 + row_number().over(Window.orderBy(col(business_col)))).cast("int").alias(key_col)
            )
        )
        incoming_df = incoming_df.join(new_values, on=business_col, how="left")

        # MERGE by business key
        delta_target = DeltaTable.forName(spark, target_table)

        (
            delta_target.alias("target")
            .merge(
                incoming_df.alias("source"),
                f"target.{business_col} = source.{business_col}"
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

        print(
            table_name,
            "incremental MERGE completed:",
            spark.table(target_table).count()
        )

else:
    raise ValueError("LOAD_TYPE must be FULL or INCREMENTAL")

# --------------------------------------------------
# 6. BUILD FACT DATAFRAME
# --------------------------------------------------

dim_farm = spark.table(f"{GOLD_SCHEMA}.dim_farm")
dim_crop = spark.table(f"{GOLD_SCHEMA}.dim_crop")
dim_season = spark.table(f"{GOLD_SCHEMA}.dim_season")
dim_irrigation = spark.table(f"{GOLD_SCHEMA}.dim_irrigation")
dim_soil = spark.table(f"{GOLD_SCHEMA}.dim_soil")

fact_df = (
    silver_df.alias("s")

    .join(
        dim_farm.alias("f"),
        col("s.farm_id") == col("f.farm_id"),
        "inner"
    )

    .join(
        dim_crop.alias("c"),
        col("s.crop_type") == col("c.crop_type"),
        "inner"
    )

    .join(
        dim_season.alias("se"),
        col("s.season") == col("se.season"),
        "inner"
    )

    .join(
        dim_irrigation.alias("i"),
        col("s.irrigation_type") == col("i.irrigation_type"),
        "inner"
    )

    .join(
        dim_soil.alias("so"),
        col("s.soil_type") == col("so.soil_type"),
        "inner"
    )

    .select(
        col("f.farm_key"),
        col("c.crop_key"),
        col("se.season_key"),
        col("i.irrigation_key"),
        col("so.soil_key"),

        col("s.farm_area_acres"),
        col("s.fertilizer_used_tons"),
        col("s.pesticide_used_kg"),
        col("s.yield_tons"),
        col("s.water_usage_cubicmeters"),

        col("s.row_hash"),
        col("s.source_name"),
        col("s.ingestion_timestamp").alias("updated_at"),
    )
)



# --------------------------------------------------
# 8. FACT FULL / INCREMENTAL LOAD
# --------------------------------------------------

if LOAD_TYPE == "FULL":

    fact_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(FACT_TABLE)

    print("Fact FULL load completed")

elif LOAD_TYPE == "INCREMENTAL":

    delta_fact = DeltaTable.forName(spark, FACT_TABLE)

    merge_condition = """
        target.farm_key = source.farm_key
        AND target.crop_key = source.crop_key
        AND target.season_key = source.season_key
        AND target.irrigation_key = source.irrigation_key
    """

    (
        delta_fact.alias("target")
        .merge(
            fact_df.alias("source"),
            merge_condition
        )
        .whenMatchedUpdateAll(
            condition="NOT (target.row_hash <=> source.row_hash)"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Fact INCREMENTAL MERGE completed")

# --------------------------------------------------
# 9. FINAL COUNTS
# --------------------------------------------------

for table_name in list(value_dimension_config.keys()) + ["dim_farm", "fact_agriculture"]:

    target_table = f"{GOLD_SCHEMA}.{table_name}"

    print(
        table_name,
        "=>",
        spark.table(target_table).count(),
        "records"
    )
